In [127]:
import os
from atproto import Client
from convenient_pickle import *
from atproto import FirehoseSubscribeReposClient, firehose_models, parse_subscribe_repos_message
from atproto import CAR, models
from atproto_client.models.network.bsky.jetstream.subscribe_events import Commit
from atproto import JetstreamClient, jetstream_models, models
import time
import threading
import queue
import base64

## Setting up bluesky password and client

In [124]:
client = Client()
bluesky_api_key = load_pickle('api_info_do_not_upload/bluesky_api_key')

In [125]:
client.login('misterscwapy.bsky.social', bluesky_api_key)

ProfileViewDetailed(did='did:plc:d5xogttjlyrqsqviaauerahn', handle='misterscwapy.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, germ=None, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:d5xogttjlyrqsqviaauerahn/bafkreicalkjv6l5viwzurde4ffbdlu5ygmwfpne4wcy7rgtn2isbbfbtci', banner=None, created_at='2024-11-23T21:45:31.945Z', debug=None, description=None, display_name='', followers_count=0, follows_count=8, indexed_at='2024-11-23T21:45:31.945Z', joined_via_starter_pack=None, labels=[], pinned_post=None, posts_count=1, pronouns=None, status=None, verification=None, viewer=ViewerState(activity_subscription=None, blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers

## Start up jetstream access

In [163]:
client = JetstreamClient(params={'kinds':['commit']})

msg_queue = queue.Queue()
outdict = dict()
messagecount = 0
stop_seconds = 1000
overtime = None


def unwrap_bytes(obj):
    if isinstance(obj, dict):
        if set(obj.keys()) == {'$bytes'}:
            return base64.b64decode(obj['$bytes'])
        return {k: unwrap_bytes(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [unwrap_bytes(v) for v in obj]
    return obj

def on_message_handler(event):
    event = unwrap_bytes(event)
    try:
        msg_queue.put(event)
    except: 
        pass

def stop_after_n_sec():
    global overtime
    time.sleep(stop_seconds)
    client.stop()
    overtime = time.time()
    msg_queue.put(None)  

def pop_em_over():
    global messagecount
    while True:
        message = msg_queue.get() 
        if message is None:
            break
        messagecount += 1
        if message.collection not in outdict.keys():
            outdict[message.collection] = [message]
        else: 
            outdict[message.collection].append(message)
    print(f"Between the client stopping and me stopping, there were {round(time.time()-overtime, 4)} seconds!")

threading.Thread(target=stop_after_n_sec).start()
threading.Thread(target=pop_em_over).start()
client.start(on_message_handler)

Between the client stopping and me stopping, there were 0.0128 seconds!


In [164]:
messagecount

440777

In [165]:
print('\n'.join([f'{i}:{len(outdict[i])}' for i in outdict.keys()]))

app.bsky.feed.like:253704
app.bsky.feed.post:105256
app.bsky.feed.repost:41158
app.bsky.feed.generator:115
app.bsky.graph.follow:26446
app.bsky.graph.block:2276
app.bsky.feed.threadgate:2498
place.stream.broadcast.origin:366
app.bsky.actor.profile:1004
community.lexicon.sports.baseball.pitcherState:148
community.lexicon.sports.baseball.batterState:57
community.lexicon.sports.baseball.gameState:125
app.bsky.feed.postgate:1284
app.bsky.actor.status:636
app.bsky.graph.listitem:1405
ai.afrilingua.feed.content:139
place.stream.livestream:379
place.stream.chat.message:84
at.freeq.deviceKey:1
at.adsb.flight.record:33
site.standard.document:154
social.coves.community.acceptance:44
social.coves.community.postv2:27
net.atmowx.observation:42
fm.teal.feed.play:47
fm.teal.actor.status:65
chat.bsky.actor.declaration:30
com.puzzmo.streak:28
app.bsky.labeler.service:14
app.bsky.graph.list:14
community.lexicon.sports.game:5
app.aozoraquest.world.npcs:1
jp.5leaf.sync.mastodon:16
app.studynext.status:6
o

In [166]:
to_make_urls = outdict['app.bsky.graph.block'][150]

print(f'bsky.app/profile/{to_make_urls.did}')
print(f'bsky.app/profile/{to_make_urls.record.subject}')

bsky.app/profile/did:plc:gq5cb5qvtl64ztjrd4f6oras
bsky.app/profile/did:plc:riejzxkdogau3cc5jc6j7jzi


In [167]:
to_make_urls = outdict['app.bsky.graph.block'][200]
to_make_urls

Commit(collection='app.bsky.graph.block', did='did:plc:bxkn42rymhy3o25klobdibkc', operation='create', rev='3mvjg6idgr52r', rkey='3mvjg6id6x52r', seq=25874486167, time='2026-09-15T00:57:28.786460Z', cid='bafyreib4wbzsy56yaulmqt6voarueu67aizpcvfppob54b5oim7oe6owmq', record=Record(created_at='2026-09-15T00:57:28.340Z', subject='did:plc:ttupzjx4nb53qa6tut6boxhn', py_type='app.bsky.graph.block'), py_type='network.bsky.jetstream.subscribeEvents#commit')

In [168]:
blockdict = dict()
for i in outdict['app.bsky.graph.block']:
    if i.record != None: 
        if i.record.subject not in blockdict.keys(): 
            blockdict[i.record.subject] = 1
        else: 
            blockdict[i.record.subject] += 1
print(sorted([(i, blockdict[i]) for i in blockdict.keys() if blockdict[i] > 3], key = lambda x: -x[1]))

[('did:plc:wwsd6jfxqzortr5gp4oiy6tx', 18), ('did:plc:szj2qzkpnhamluv4zsye443r', 17), ('did:plc:kvuaqglv7jzo6fqlhiikygee', 8), ('did:plc:ldqprbxpwwysfewq7z2j5u7l', 7), ('did:plc:k564z24vswgzl6q47ixc6bof', 6), ('did:plc:cshkhphcdgv4z6w2pwgcgzlm', 6), ('did:plc:wxnfg2xdph2pbxbybsq7cw6s', 5), ('did:plc:64lthwjlbfnv62wwwkydi4w5', 5), ('did:plc:xgufx6jv37jwlgxyzl243vk4', 5), ('did:plc:cyxiz4fwnpe43dzuqu546blr', 5), ('did:plc:iwanz3gbp32oz4mj7wwhyd6n', 4), ('did:plc:q5pmpnuhiw2eqd3hz7zvj64q', 4), ('did:plc:z3iy2wd46tmuobpex4i3ph2m', 4), ('did:plc:sb6fu4sinwphqpvoznvz7efo', 4), ('did:plc:jqxufbk3cuvbii5prxcls2mk', 4), ('did:plc:3uvhblt5c4ytjdxliscyhg42', 4), ('did:plc:sp2cqybuhdll6qbdsi6asqaz', 4), ('did:plc:oc6n653yetcmzbucqtahv24p', 4)]
